In [0]:
"""
BOOTSTRAP_SERVERS='pkc-15v0n6.us-east5.gcp.confluent.cloud:9092'
API_KEY='NXI57PPBTORUVM6Y'
API_SECRET='cflt0HO0DDLFFhJop/JaAEWr49w7/TeQZssJ9bWBSEtmQk9d9MNc4rbDhTzpdinw'
TOPIC_NAME='credit_card_transaction'
"""

In [0]:
import json
kafka_connection_json = dbutils.secrets.get(scope="finguard-scope", key="kafka_connection_details")
kafka_config = json.loads(kafka_connection_json)
BOOTSTRAP_SERVERS=kafka_config['kafka_bootstrap_servers']
API_KEY=kafka_config['kafka_api_key']
API_SECRET=kafka_config['kafka_api_secret']
TOPIC_NAME=kafka_config['kafka_topic']

In [0]:
jaas_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{API_KEY}" password="{API_SECRET}";'

In [0]:
sample_batch = (spark.read.format("kafka")
           .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
           .option("subscribe", TOPIC_NAME) 
           .option("kafka.security.protocol","SASL_SSL")
           .option("kafka.sasl.mechanism", "PLAIN")
           .option("kafka.sasl.jaas.config", jaas_config)
           .option("startingOffsets", "earliest")
           .load()
)

In [0]:
sample_batch.count()

In [0]:
display(sample_batch)

In [0]:

from pyspark.sql.functions import col
parsed_batch = sample_batch.select(col("key").cast("string"), col("value").cast("string"), col("topic"), col("partition"), col("offset"), col("timestamp"), col("timestampType"))
display(parsed_batch)

In [0]:
parsed_batch.write.saveAsTable("finguard.bronze.transactions_batch_test")

In [0]:
streaming_df = (spark.readStream.format("kafka")
           .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
           .option("subscribe", TOPIC_NAME) 
           .option("kafka.security.protocol","SASL_SSL")
           .option("kafka.sasl.mechanism", "PLAIN")
           .option("kafka.sasl.jaas.config", jaas_config)
           .option("startingOffsets", "earliest")
           .load()
)

In [0]:
from pyspark.sql.functions import col
parsed_streaming_df = streaming_df.select(col("key").cast("string"), col("value").cast("string"), col("topic"), col("partition"), col("offset"), col("timestamp"), col("timestampType"))

In [0]:
streaming_query = (parsed_streaming_df.writeStream.format("delta")
.outputMode("append")
.option("checkpointLocation", "/Volumes/finguard/source/transactions/checkpoint/")
.trigger(availableNow=True)
.table("finguard.bronze.transactions_streaming_test")
)

print("Query Started with query id:", streaming_query.id)

In [0]:
%sql  
 select * from finguard.bronze.transactions_streaming_test